# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [00:58<00:00, 19.48s/it]


In [3]:
len(deals)

30

In [4]:
deals[10].describe()

"Title: ZAGG Universal Bluetooth Keyboard & Stand for $15 + free shipping\nDetails: That's a $25 savings. Buy Now at eBay\nFeatures: \nURL: https://www.dealnews.com/ZAGG-Universal-Bluetooth-Keyboard-Stand-for-15-free-shipping/21806930.html?iref=rss-c39"

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [5]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [6]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [7]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Refurb Apple iPhone 15 256GB Smartphone for $386 + free shipping
Details: That's $514 off and the best deal we've seen on this model. A 1-year Allstate warranty applies. Buy Now at eBay
Features: 
URL: https://www.dealnews.com/Refurb-Apple-iPhone-15-256-GB-Smartphone-for-386-free-shipping/21806918.html?iref=rss-c142

Title: Best Buy Outlet Event: Up to 60% off + free shipping
D

In [8]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='Refurbished Apple iPhone 15 with 256GB storage offering the core iPhone experience in a modern design. This unit includes a full-size OLED display, Apple’s A16/A17-class performance (depending on refurb spec), Face ID biometric security, and a capable camera system for high-quality photos and video. It’s refurbished and sold with a 1-year Allstate warranty, making it a lower-cost alternative to new models while retaining Apple ecosystem compatibility.', price=386.0, url='https://www.dealnews.com/Refurb-Apple-iPhone-15-256-GB-Smartphone-for-386-free-shipping/21806918.html?iref=rss-c142'), Deal(product_description='Kospet Tank T4C rugged smartwatch built for outdoor and heavy-duty use, featuring a 1.5" high-brightness AMOLED round display and up to 15 days of battery life. It includes dual-band six-system GNSS for precise tracking, MIL-STD-810H military-grade durability, IP69K and 5 ATM water resistance, an integrated five-level pro flashlig

In [9]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


Refurbished Apple iPhone 15 with 256GB storage offering the core iPhone experience in a modern design. This unit includes a full-size OLED display, Apple’s A16/A17-class performance (depending on refurb spec), Face ID biometric security, and a capable camera system for high-quality photos and video. It’s refurbished and sold with a 1-year Allstate warranty, making it a lower-cost alternative to new models while retaining Apple ecosystem compatibility.
386.0
https://www.dealnews.com/Refurb-Apple-iPhone-15-256-GB-Smartphone-for-386-free-shipping/21806918.html?iref=rss-c142

Kospet Tank T4C rugged smartwatch built for outdoor and heavy-duty use, featuring a 1.5" high-brightness AMOLED round display and up to 15 days of battery life. It includes dual-band six-system GNSS for precise tracking, MIL-STD-810H military-grade durability, IP69K and 5 ATM water resistance, an integrated five-level pro flashlight, and a built-in walkie-talkie with roughly a 40m range for direct-device communication

In [10]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [11]:
from agents.scanner_agent import ScannerAgent

In [12]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [13]:
result

DealSelection(deals=[Deal(product_description="Refurbished Apple iPhone 15 with 256GB of internal storage. This smartphone includes Apple's latest design and performance features from the iPhone 15 line, offering a large internal capacity for apps, photos, and media. The unit is sold refurbished and comes with a 1-year Allstate warranty, making it a cost-effective option for buyers who want near-new hardware at a reduced price.", price=386.0, url='https://www.dealnews.com/Refurb-Apple-iPhone-15-256-GB-Smartphone-for-386-free-shipping/21806918.html?iref=rss-c142'), Deal(product_description='Kospet Tank T4C rugged smartwatch featuring a 1.5" AMOLED high-brightness round display, up to 15 days of battery life, and MIL-STD-810H military-grade durability. It offers advanced outdoor features including dual-band six-system GNSS tracking, IP69K and 5 ATM water resistance, an integrated 5-level professional flashlight, and a built-in walkie-talkie with about a 40m range for short-range voice co

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [14]:
load_dotenv(override=True)

True

In [15]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [16]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user not found
Pushover token not found


In [17]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [18]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [19]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


In [20]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

INFO:root:[Messaging Agent] Messaging Agent is using Claude to craft the message
16:51:28 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= claude-sonnet-4-5; provider = anthropic
INFO:LiteLLM:
LiteLLM completion() model= claude-sonnet-4-5; provider = anthropic


AuthenticationError: litellm.AuthenticationError: Missing Anthropic API Key - A call is being made to anthropic but no key is set either in the environment variables or via params. Please set `ANTHROPIC_API_KEY` in your environment vars